# Oráculo do ScribblePrompt com negativo do Cellpose

**Objetivo:** medir o que o ScribblePrompt congelado consegue entregar quando recebe marcadores **conhecidos** (tirados do GT
ou do Cellpose), e se um canal negativo derivado do Cellpose segura a segmentação dentro dos núcleos. Não há MarkerUNet nem
treino: cada configuração é uma única passada da rede.

| Eixo | Valores |
|---|---|
| Marcador positivo | GT inteiro, miolo de cada núcleo, centros, máscara do Cellpose, miolo de cada instância do Cellpose |
| Canal negativo | zero, complemento do positivo, fundo do Cellpose a 3 e a 8 px, fundo do Cellpose esparso, fundo do GT a 3 px |
| Entrada do ScribblePrompt | 128² e 256² |
| Pós-processamento | limiar de saída 0,5–0,9; recorte da saída pela máscara do Cellpose dilatada (nenhum, 3 px, 8 px) |
| Imagens | as 37 de treino (a escolha de configuração não usa o teste); as três em 20× aparecem separadas |
| Linha de base | Cellpose sozinho, nas mesmas imagens |

**Saída:** `docs/estudo/resultados/oraculo_negativo/` (`por_imagem.csv.gz` com uma linha por imagem e configuração,
`resumo.csv` com as médias e `config.json` com a grade e a procedência). Roda em CPU em ~20 min (12 núcleos).

## 0. Setup

In [1]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Pedro-io/cell-fuzzy-seg.git"
BRANCH = "homolog"


def find_repo_root(start="."):
    p = os.path.abspath(start)
    prev = None
    while p != prev:
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "pyproject.toml")):
            return p
        prev, p = p, os.path.dirname(p)
    return None


if "google.colab" in sys.modules:
    repo_root = os.path.join("/content", "cell-fuzzy-seg")
    if os.path.exists(repo_root):
        subprocess.run(["git", "-C", repo_root, "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", repo_root, "pull", "--ff-only", "origin", BRANCH], check=True)
    else:
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, repo_root], check=True)
else:
    repo_root = find_repo_root(os.getcwd())
    if repo_root is None:
        raise RuntimeError("Raiz do repositório não encontrada a partir do diretório atual.")

os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Repositório:", repo_root)
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip())

Repositório: /content/cell-fuzzy-seg
d90ce99 feat(experimentos): varredura de perdas e pesos só com validação cruzada


In [2]:
if "google.colab" in sys.modules:
    get_ipython().system("python -m pip install -r requirements.txt -q")
else:
    print("Ambiente local: verifique se as dependências do requirements.txt estão instaladas.")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 23.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 69.9 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 16.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 61.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 34.9 MB/s eta 0:00:0000:0100:01


## 1. Configuração

In [3]:
import hashlib
import json
import time

import numpy as np
import pandas as pd
import torch
from scipy import ndimage

from src.data.load.preprocessed_dataset import load_preprocessed
from src.evaluation.metrics import binary_metrics
from src.evaluation.report import provenance
from src.models.networks.final_segmentation.scribble_prompting_network import ScribblePromptingNetwork
from src.pipeline.steps.preprocessing.distance_map_step import compute_instance_distance_map

OUTPUT_DIR = os.path.join("docs", "estudo", "resultados", "oraculo_negativo")
MODELS_DIR = os.path.expanduser(os.path.join("~", "cell-fuzzy-models"))
SPLIT = "train"
RESOLUCOES = [(128, 128), (256, 256)]
LIMIARES = [0.5, 0.6, 0.7, 0.8, 0.9]
RAIOS_RECORTE = [None, 3, 8]          # saída multiplicada pela máscara do Cellpose dilatada (None: sem recorte)
IMAGENS_20X = {"TCGA-HE-7128-01Z-00-DX1", "TCGA-HE-7129-01Z-00-DX1", "TCGA-HE-7130-01Z-00-DX1"}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

torch.set_num_threads(os.cpu_count())

## 2. Dados

In [4]:
samples = load_preprocessed(SPLIT)
print(len(samples), "imagens")


def disco(raio):
    y, x = np.ogrid[-raio:raio + 1, -raio:raio + 1]
    return x * x + y * y <= raio * raio


def dilatar(mascara, raio):
    return ndimage.binary_dilation(mascara, structure=disco(raio)) if raio > 0 else mascara

37 imagens


## 3. Marcadores positivos

- `gt`: o GT inteiro.
- `miolo`: pixels do núcleo com mapa de distância ≤ 0,5 (o mapa é normalizado por núcleo: 0 no centro, 1 na borda).
- `centros`: disco de raio 4 nos pixels de mapa 0 (o ponto mais interno de cada núcleo), cortado pelo GT.
- `cellpose`: a máscara do Cellpose.
- `cellpose_miolo`: o miolo de cada instância do Cellpose (mesma regra do `miolo`). Não usa o GT: é uma linha de base que não
  precisa de treino.

In [5]:
def marcadores(sample):
    gt = sample["ground_truth"] > 0
    dmap = sample["distance_map"]
    cp = sample["cellpose_segmentation"]
    cp_dmap = compute_instance_distance_map(cp)
    centros = gt & (dmap <= 0.0)
    return {
        "gt": gt,
        "miolo": gt & (dmap <= 0.5),
        "centros": dilatar(centros, 4) & gt,
        "cellpose": cp > 0,
        "cellpose_miolo": (cp > 0) & (cp_dmap <= 0.5),
    }

## 4. Canais negativos

- `zero`: nenhum negativo (o modo `positive` do pipeline).
- `complemento`: `1 − positivo` (o negativo denso do modo `sharpened`, sem o *sharpening*).
- `cp_fundo_r3`, `cp_fundo_r8`: fundo do Cellpose, a mais de 3 ou 8 px das máscaras.
- `cp_fundo_esparso`: pontos de raio 3 numa grade de 25 px, só onde `cp_fundo_r8` é 1.
- `gt_fundo_r3`: fundo do GT a mais de 3 px (teto: um negativo "perfeito").

In [6]:
GRADE = np.zeros((1000, 1000), bool)
GRADE[12::25, 12::25] = True
GRADE = dilatar(GRADE, 3)


def negativos(sample, positivo):
    gt = sample["ground_truth"] > 0
    cp = sample["cellpose_segmentation"] > 0
    cp_fundo_r8 = ~dilatar(cp, 8)
    return {
        "zero": np.zeros_like(positivo),
        "complemento": ~positivo,
        "cp_fundo_r3": ~dilatar(cp, 3),
        "cp_fundo_r8": cp_fundo_r8,
        "cp_fundo_esparso": cp_fundo_r8 & GRADE[: gt.shape[0], : gt.shape[1]],
        "gt_fundo_r3": ~dilatar(gt, 3),
    }

## 5. ScribblePrompt e varredura

In [7]:
checkpoint = ScribblePromptingNetwork.download_checkpoint(os.path.join(MODELS_DIR, "scribbleprompt"))
with open(checkpoint, "rb") as f:
    CHECKPOINT_SHA256 = hashlib.sha256(f.read()).hexdigest()
net = ScribblePromptingNetwork(checkpoint=checkpoint, device=DEVICE)


def segmentar(image, positivo, negativo, resolucao):
    net.input_size = tuple(resolucao)
    scribbles = np.stack([positivo, negativo], axis=-1).astype(np.float32)   # (H, W, 2) → [pos, neg]
    return net.predict({"image": image, "scribbles": scribbles})[0, 0].cpu().numpy()

18:00:09 | [ScribblePromptingNetwork] Baixando checkpoint de https://www.dropbox.com/scl/fi/pnw88n05irnv5z1snlklr/ScribblePrompt_unet_v1_nf192_res128.pt?rlkey=dr8xvkf0wj2r082h1zzpcmz5o&dl=1
18:00:12 | [ScribblePromptingNetwork] Checkpoint salvo em /root/cell-fuzzy-models/scribbleprompt/ScribblePrompt_unet_v1_nf192_res128.pt


In [8]:
linhas = []
inicio = time.time()
for n, (image_id, sample) in enumerate(samples.items(), start=1):
    gt = sample["ground_truth"]
    cp = sample["cellpose_segmentation"] > 0
    base = binary_metrics(cp, gt)
    recortes = {r: (np.ones_like(cp) if r is None else dilatar(cp, r)) for r in RAIOS_RECORTE}
    for nome_m, positivo in marcadores(sample).items():
        for nome_n, negativo in negativos(sample, positivo).items():
            for resolucao in RESOLUCOES:
                prob = segmentar(sample["image"], positivo, negativo, resolucao)
                for raio, recorte in recortes.items():
                    for limiar in LIMIARES:
                        m = binary_metrics(prob * recorte, gt, limiar)
                        linhas.append({
                            "id": image_id, "aumento_20x": image_id in IMAGENS_20X,
                            "marcador": nome_m, "negativo": nome_n, "resolucao": resolucao[0],
                            "recorte_cellpose": "nenhum" if raio is None else f"r{raio}", "limiar": limiar,
                            "area_marcador": float(positivo.mean()), "area_negativo": float(negativo.mean()),
                            **m, **{f"cellpose_{k}": v for k, v in base.items()},
                        })
    print(f"{n}/{len(samples)} {image_id} ({time.time() - inicio:.0f} s)", flush=True)

df = pd.DataFrame(linhas)

1/37 TCGA-18-5592-01Z-00-DX1 (17 s)
2/37 TCGA-21-5784-01Z-00-DX1 (32 s)
3/37 TCGA-21-5786-01Z-00-DX1 (47 s)
4/37 TCGA-38-6178-01Z-00-DX1 (63 s)
5/37 TCGA-49-4488-01Z-00-DX1 (78 s)
6/37 TCGA-50-5931-01Z-00-DX1 (93 s)
7/37 TCGA-A7-A13E-01Z-00-DX1 (108 s)
8/37 TCGA-A7-A13F-01Z-00-DX1 (123 s)
9/37 TCGA-AR-A1AK-01Z-00-DX1 (139 s)
10/37 TCGA-AR-A1AS-01Z-00-DX1 (154 s)
11/37 TCGA-AY-A8YK-01A-01-TS1 (169 s)
12/37 TCGA-B0-5698-01Z-00-DX1 (184 s)
13/37 TCGA-B0-5710-01Z-00-DX1 (200 s)
14/37 TCGA-B0-5711-01Z-00-DX1 (216 s)
15/37 TCGA-BC-A217-01Z-00-DX1 (230 s)
16/37 TCGA-CH-5767-01Z-00-DX1 (245 s)
17/37 TCGA-DK-A2I6-01A-01-TS1 (260 s)
18/37 TCGA-E2-A14V-01Z-00-DX1 (276 s)
19/37 TCGA-E2-A1B5-01Z-00-DX1 (292 s)
20/37 TCGA-F9-A8NY-01Z-00-DX1 (306 s)
21/37 TCGA-FG-A87N-01Z-00-DX1 (321 s)
22/37 TCGA-G2-A2EK-01A-02-TSB (336 s)
23/37 TCGA-G9-6336-01Z-00-DX1 (351 s)
24/37 TCGA-G9-6348-01Z-00-DX1 (367 s)
25/37 TCGA-G9-6356-01Z-00-DX1 (381 s)
26/37 TCGA-G9-6362-01Z-00-DX1 (396 s)
27/37 TCGA-G9-6363-01Z-00-D

## 6. Registro

In [9]:
os.makedirs(OUTPUT_DIR, exist_ok=False)
df.to_csv(os.path.join(OUTPUT_DIR, "por_imagem.csv.gz"), index=False)
config = {
    "split": SPLIT, "n_imagens": len(samples), "resolucoes": RESOLUCOES, "limiares": LIMIARES,
    "raios_recorte": RAIOS_RECORTE, "imagens_20x": sorted(IMAGENS_20X),
    "checkpoint_sha256": CHECKPOINT_SHA256,
    "procedencia": provenance(), "duracao_s": round(time.time() - inicio, 1),
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

## 7. Resumo

Média por imagem de cada configuração; `dif_dice` é a diferença pareada contra o Cellpose, e `n_melhor`, o número de imagens em
que a configuração supera o Cellpose.

In [10]:
chave = ["marcador", "negativo", "resolucao", "recorte_cellpose", "limiar"]
df["dif_dice"] = df["dice"] - df["cellpose_dice"]
resumo = df.groupby(chave).agg(
    dice=("dice", "mean"), precision=("precision", "mean"), recall=("recall", "mean"),
    mass_ratio=("mass_ratio", "mean"), dif_dice=("dif_dice", "mean"),
    n_melhor=("dif_dice", lambda d: int((d > 0).sum())),
).reset_index()
resumo_20x = df[df["aumento_20x"]].groupby(chave)["dice"].mean().rename("dice_20x")
resumo = resumo.join(resumo_20x, on=chave)
resumo.to_csv(os.path.join(OUTPUT_DIR, "resumo.csv"), index=False)

cellpose = df.drop_duplicates("id")
print("Cellpose:", cellpose[["cellpose_dice", "cellpose_precision", "cellpose_recall", "cellpose_mass_ratio"]].mean().round(3).to_dict())
pd.set_option("display.width", 200)

Cellpose: {'cellpose_dice': 0.845, 'cellpose_precision': 0.867, 'cellpose_recall': 0.826, 'cellpose_mass_ratio': 0.955}


Melhor configuração de cada par marcador × negativo (sobre resolução, recorte e limiar):

In [11]:
melhores = resumo.loc[resumo.groupby(["marcador", "negativo"])["dice"].idxmax()]
melhores.sort_values(["marcador", "dice"], ascending=[True, False]).round(3)

,marcador,negativo,resolucao,recorte_cellpose,limiar,dice,precision,recall,mass_ratio,dif_dice,n_melhor,dice_20x
137,cellpose,gt_fundo_r3,256,nenhum,0.7,0.879,0.861,0.899,1.045,0.034,35,0.842
20,cellpose,complemento,256,r3,0.5,0.837,0.833,0.844,1.017,-0.008,5,0.817
82,cellpose,cp_fundo_r3,256,r3,0.7,0.832,0.810,0.859,1.064,-0.013,6,0.807
173,cellpose,zero,256,r3,0.8,0.826,0.824,0.832,1.013,-0.019,0,0.790
113,cellpose,cp_fundo_r8,256,r3,0.8,0.824,0.800,0.852,1.069,-0.021,2,0.793
53,cellpose,cp_fundo_esparso,256,r3,0.8,0.822,0.800,0.847,1.063,-0.023,0,0.793
315,cellpose_miolo,gt_fundo_r3,256,nenhum,0.5,0.838,0.888,0.799,0.903,-0.007,17,0.817
260,cellpose_miolo,cp_fundo_r3,256,r3,0.5,0.797,0.839,0.765,0.917,-0.048,0,0.786
290,cellpose_miolo,cp_fundo_r8,256,r3,0.5,0.762,0.828,0.712,0.868,-0.083,0,0.763
230,cellpose_miolo,cp_fundo_esparso,256,r3,0.5,0.741,0.823,0.687,0.850,-0.104,0,0.746


Dice por marcador e negativo, em 256², limiar 0,5 e sem recorte:

In [12]:
sel = resumo[(resumo.resolucao == 256) & (resumo.recorte_cellpose == "nenhum") & (resumo.limiar == 0.5)]
sel.pivot(index="marcador", columns="negativo", values="dice").round(3)

negativo,complemento,cp_fundo_esparso,cp_fundo_r3,cp_fundo_r8,gt_fundo_r3,zero
marcador,,,,,,
cellpose,0.837,0.597,0.808,0.763,0.847,0.795
cellpose_miolo,0.514,0.554,0.797,0.752,0.838,0.679
centros,0.306,0.509,0.786,0.707,0.830,0.542
gt,0.943,0.608,0.841,0.784,0.870,0.839
miolo,0.561,0.592,0.820,0.787,0.870,0.754


Sem negativo (256², sem recorte): Dice e massa por limiar de saída.

In [13]:
sel = resumo[(resumo.negativo == "zero") & (resumo.resolucao == 256) & (resumo.recorte_cellpose == "nenhum")]
display(sel.pivot(index="marcador", columns="limiar", values="dice").round(3))
sel.pivot(index="marcador", columns="limiar", values="mass_ratio").round(2)

limiar,0.5,0.6,0.7,0.8,0.9
marcador,,,,,
cellpose,0.795,0.810,0.821,0.825,0.809
cellpose_miolo,0.679,0.635,0.583,0.515,0.398
centros,0.542,0.483,0.420,0.346,0.237
gt,0.839,0.870,0.897,0.915,0.903
miolo,0.754,0.704,0.643,0.564,0.430


limiar,0.5,0.6,0.7,0.8,0.9
marcador,,,,,
cellpose,1.28,1.20,1.11,1.02,0.88
cellpose_miolo,0.61,0.54,0.46,0.38,0.27
centros,0.43,0.35,0.29,0.22,0.14
gt,1.38,1.28,1.19,1.08,0.92
miolo,0.65,0.56,0.48,0.40,0.28


Imagens em 20× contra as demais, em algumas configurações:

In [14]:
CONFIGS = [
    ("gt", "zero", 256, "nenhum", 0.8),
    ("miolo", "zero", 256, "nenhum", 0.5),
    ("miolo", "cp_fundo_r3", 256, "nenhum", 0.5),
    ("miolo", "zero", 128, "nenhum", 0.5),
    ("miolo", "cp_fundo_r3", 128, "nenhum", 0.5),
    ("cellpose_miolo", "cp_fundo_r3", 256, "r3", 0.5),
]
tabela = []
for marcador, negativo, resolucao, recorte, limiar in CONFIGS:
    s = df[(df.marcador == marcador) & (df.negativo == negativo) & (df.resolucao == resolucao)
           & (df.recorte_cellpose == recorte) & (df.limiar == limiar)]
    for aumento, g in s.groupby("aumento_20x"):
        tabela.append({"config": f"{marcador} / {negativo} / {resolucao}² / {recorte} / {limiar}",
                       "20x": aumento, **g[["dice", "precision", "recall", "mass_ratio", "cellpose_dice"]].mean().round(3)})
pd.DataFrame(tabela)

,config,20x,dice,precision,recall,mass_ratio,cellpose_dice
0,gt / zero / 256² / nenhum / 0.8,False,0.919,0.885,0.957,1.081,0.846
1,gt / zero / 256² / nenhum / 0.8,True,0.872,0.853,0.893,1.048,0.836
2,miolo / zero / 256² / nenhum / 0.5,False,0.751,0.965,0.618,0.641,0.846
3,miolo / zero / 256² / nenhum / 0.5,True,0.788,0.949,0.674,0.711,0.836
4,miolo / cp_fundo_r3 / 256² / nenhum / 0.5,False,0.820,0.861,0.789,0.922,0.846
5,miolo / cp_fundo_r3 / 256² / nenhum / 0.5,True,0.823,0.821,0.826,1.006,0.836
6,miolo / zero / 128² / nenhum / 0.5,False,0.677,0.933,0.536,0.581,0.846
7,miolo / zero / 128² / nenhum / 0.5,True,0.542,0.806,0.410,0.509,0.836
8,miolo / cp_fundo_r3 / 128² / nenhum / 0.5,False,0.770,0.885,0.686,0.778,0.846
9,miolo / cp_fundo_r3 / 128² / nenhum / 0.5,True,0.659,0.796,0.562,0.708,0.836


## 8. Figuras

Duas imagens: a de 40× com o Dice do Cellpose mais próximo da mediana das imagens fora de 20×, e uma das imagens em 20×. Em
cada uma, um recorte de 200×200 px (a janela com mais núcleo, numa grade de 100 px). Cores: positivo em azul, negativo em
laranja; nos mapas de erro, acerto em cinza, falso positivo em verde-água e falso negativo em violeta.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from skimage.segmentation import find_boundaries

COR_POS = np.array([0x2A, 0x78, 0xD6]) / 255
COR_NEG = np.array([0xEB, 0x68, 0x34]) / 255
COR_VP = np.array([0xB8, 0xB7, 0xB0]) / 255
COR_FP = np.array([0x1B, 0xAF, 0x7A]) / 255
COR_FN = np.array([0x4A, 0x3A, 0xA7]) / 255
LADO = 200

dice_cp = {i: binary_metrics(s["cellpose_segmentation"] > 0, s["ground_truth"])["dice"] for i, s in samples.items()}
ids_40x = sorted((i for i in samples if i not in IMAGENS_20X), key=dice_cp.get)
FIG_IDS = {"40×": ids_40x[len(ids_40x) // 2], "20×": "TCGA-HE-7128-01Z-00-DX1"}


def janela(gt, lado=LADO, passo=100):
    """Canto (y, x) da janela lado×lado com mais pixels de núcleo."""
    h, w = gt.shape
    cantos = [(y, x) for y in range(0, h - lado + 1, passo) for x in range(0, w - lado + 1, passo)]
    return max(cantos, key=lambda c: gt[c[0]:c[0] + lado, c[1]:c[1] + lado].sum())


def sobrepor(image, camadas, alfa=0.6):
    out = image.astype(float) / 255
    for mascara, cor in camadas:
        out[mascara] = (1 - alfa) * out[mascara] + alfa * cor
    return out


def mapa_erro(pred, gt):
    out = np.ones(gt.shape + (3,))
    out[pred & gt] = COR_VP
    out[pred & ~gt] = COR_FP
    out[~pred & gt] = COR_FN
    return out


RECORTES = {}
for nome, image_id in FIG_IDS.items():
    y, x = janela(samples[image_id]["ground_truth"] > 0)
    RECORTES[nome] = (image_id, (slice(y, y + LADO), slice(x, x + LADO)))
print({k: (v[0], v[1][0].start, v[1][1].start) for k, v in RECORTES.items()})

**Entradas.** O que cada marcador positivo e cada negativo marcam. A faixa sem cor entre o azul e o laranja é a margem
livre de 3 ou 8 px em volta das máscaras do Cellpose.

In [ ]:
colunas = ["imagem", "GT", "Cellpose", "miolo", "centros", "miolo + fundo do\nCellpose r3",
           "miolo + fundo do\nCellpose r8", "miolo + fundo do\nCellpose esparso"]
fig, axes = plt.subplots(2, len(colunas), figsize=(2.3 * len(colunas), 5.2))
for linha, (nome, (image_id, fatia)) in enumerate(RECORTES.items()):
    s = samples[image_id]
    pos = marcadores(s)
    neg = negativos(s, pos["miolo"])
    img = s["image"][fatia]
    paineis = [
        img / 255,
        np.where((s["ground_truth"] > 0)[fatia][..., None], 0.25, 1.0) * np.ones(3),
        np.where((s["cellpose_segmentation"] > 0)[fatia][..., None], 0.25, 1.0) * np.ones(3),
        sobrepor(img, [(pos["miolo"][fatia], COR_POS)]),
        sobrepor(img, [(pos["centros"][fatia], COR_POS)]),
        sobrepor(img, [(pos["miolo"][fatia], COR_POS), (neg["cp_fundo_r3"][fatia], COR_NEG)]),
        sobrepor(img, [(pos["miolo"][fatia], COR_POS), (neg["cp_fundo_r8"][fatia], COR_NEG)]),
        sobrepor(img, [(pos["miolo"][fatia], COR_POS), (neg["cp_fundo_esparso"][fatia], COR_NEG)]),
    ]
    for j, painel in enumerate(paineis):
        ax = axes[linha, j]
        ax.imshow(painel, interpolation="nearest")
        ax.set_xticks([]), ax.set_yticks([])
        if linha == 0:
            ax.set_title(colunas[j], fontsize=9)
    axes[linha, 0].set_ylabel(f"{nome}\n{image_id[:12]}", fontsize=9)
fig.legend(handles=[Patch(color=COR_POS, label="positivo"), Patch(color=COR_NEG, label="negativo")],
           loc="lower center", ncol=2, frameon=False, fontsize=9)
fig.suptitle(f"Entradas do oráculo (recortes de {LADO}×{LADO} px da imagem de 1000×1000)", fontsize=10)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(os.path.join(OUTPUT_DIR, "figura_entradas.png"), dpi=100)
plt.show()

**Resolução.** A imagem (em tons de cinza, como o ScribblePrompt a recebe) e o miolo, no tamanho original e depois da redução
para 256² e 128², feita pelo próprio pacote (bilinear, sem antialias). O recorte de 200 px vira 51 px em 256² e 26 px em 128².

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(7.5, 5.4))
for linha, (nome, (image_id, fatia)) in enumerate(RECORTES.items()):
    s = samples[image_id]
    cinza = net._prepare_image(s["image"], "cpu")
    pos = torch.from_numpy(marcadores(s)["miolo"].astype(np.float32))[None, None]
    scribbles = torch.cat([pos, torch.zeros_like(pos)], dim=1)
    for j, tamanho in enumerate([None, (256, 256), (128, 128)]):
        if tamanho is None:
            img_r, pos_r, f = cinza[0, 0].numpy(), pos[0, 0].numpy(), 1.0
        else:
            r = net._rescale_inputs({"img": cinza, "scribbles": scribbles}, tamanho)
            img_r, pos_r, f = r["img"][0, 0].numpy(), r["scribbles"][0, 0].numpy(), tamanho[0] / 1000
        fy = slice(int(fatia[0].start * f), int(fatia[0].stop * f))
        fx = slice(int(fatia[1].start * f), int(fatia[1].stop * f))
        rgb = np.repeat(img_r[fy, fx][..., None], 3, axis=2)
        a = 0.6 * np.clip(pos_r[fy, fx], 0, 1)[..., None]
        ax = axes[linha, j]
        ax.imshow((1 - a) * rgb + a * COR_POS, interpolation="nearest")
        ax.set_xticks([]), ax.set_yticks([])
        if linha == 0:
            ax.set_title("original (1000²)" if tamanho is None else f"entrada {tamanho[0]}²", fontsize=9)
    axes[linha, 0].set_ylabel(f"{nome}\n{image_id[:12]}", fontsize=9)
fig.suptitle("O mesmo recorte no tamanho original e como o ScribblePrompt o recebe", fontsize=10)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "figura_resolucao.png"), dpi=100)
plt.show()

**Saídas.** Mapas de erro em 256², limiar 0,5, sem recorte. O Dice no título é o da imagem inteira.

In [ ]:
SAIDAS = [
    ("miolo", "zero", "miolo,\nsem negativo"),
    ("miolo", "cp_fundo_r3", "miolo + fundo do\nCellpose r3"),
    ("miolo", "gt_fundo_r3", "miolo + fundo do\nGT r3 (teto)"),
    ("cellpose", "complemento", "Cellpose +\ncomplemento"),
]


def virgula(valor):
    return f"{valor:.3f}".replace(".", ",")


fig, axes = plt.subplots(2, len(SAIDAS) + 2, figsize=(2.4 * (len(SAIDAS) + 2), 6.4))
for linha, (nome, (image_id, fatia)) in enumerate(RECORTES.items()):
    s = samples[image_id]
    gt = s["ground_truth"] > 0
    cp = s["cellpose_segmentation"] > 0
    pos, contorno = marcadores(s), find_boundaries(gt[fatia], mode="inner")
    titulos = ["imagem +\ncontorno do GT", "Cellpose\nsozinho"] + [rotulo for _, _, rotulo in SAIDAS]
    paineis = [sobrepor(s["image"][fatia], [(contorno, np.zeros(3))], alfa=1.0), mapa_erro(cp[fatia], gt[fatia])]
    dices = ["", virgula(binary_metrics(cp, gt)["dice"])]
    for nome_m, nome_n, _ in SAIDAS:
        positivo = pos[nome_m]
        prob = segmentar(s["image"], positivo, negativos(s, positivo)[nome_n], (256, 256))
        paineis.append(mapa_erro((prob >= 0.5)[fatia], gt[fatia]))
        dices.append(virgula(binary_metrics(prob, gt)["dice"]))
    for j, (painel, dice) in enumerate(zip(paineis, dices)):
        ax = axes[linha, j]
        ax.imshow(painel, interpolation="nearest")
        ax.set_xticks([]), ax.set_yticks([])
        topo = titulos[j] + "\n" if linha == 0 else ""
        ax.set_title(topo + (f"Dice {dice}" if dice else ""), fontsize=9)
    axes[linha, 0].set_ylabel(f"{nome}\n{image_id[:12]}", fontsize=9)
fig.legend(handles=[Patch(color=COR_VP, label="acerto (núcleo)"), Patch(color=COR_FP, label="falso positivo"),
                    Patch(color=COR_FN, label="falso negativo")], loc="lower center", ncol=3, frameon=False, fontsize=9)
fig.suptitle("Saídas do ScribblePrompt (256², limiar 0,5) e do Cellpose; Dice da imagem inteira", fontsize=10)
fig.tight_layout(rect=(0, 0.04, 1, 1))
fig.savefig(os.path.join(OUTPUT_DIR, "figura_saidas.png"), dpi=100)
plt.show()